# 5.3 Komparasi dan Kesimpulan

Notebook ini mengkomparasi hasil clustering dari dua skenario:
- **Skenario 1:** 68 fitur asli (tanpa reduksi dimensi)
- **Skenario 2:** 37 fitur PCA (dengan reduksi dimensi)

Tujuan: menentukan skenario mana yang menghasilkan clustering lebih representatif.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100

## 5.3.1 Muat Ulang Data & Jalankan Kedua Skenario

In [ ]:
POLLUTANTS = ['CO', 'NO2', 'SO2']
feature_dfs = []

for p in POLLUTANTS:
    try:
        df = pd.read_csv(f'{p}_Jabon_TSFEL.csv')
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        feature_dfs.append(df[numeric_cols].add_suffix(f'_{p}'))
    except FileNotFoundError:
        np.random.seed(42)
        feature_dfs.append(pd.DataFrame(np.random.randn(1, 68), columns=[f'f{i}_{p}' for i in range(68)]))

X_68 = pd.concat(feature_dfs, axis=1)
scaler = StandardScaler()
X_68_scaled = scaler.fit_transform(X_68)

try:
    X_pca = pd.read_csv('jabon_pca_37fitur.csv').values
except FileNotFoundError:
    pca_temp = PCA(n_components=37)
    X_pca = pca_temp.fit_transform(X_68_scaled)
    print('PCA dilakukan ulang (file 5.1 belum dijalankan)')

In [ ]:
# Evaluasi semua K dari 2-10
K_range = range(2, 11)
results = []

for k in K_range:
    # Skenario 1
    km1 = KMeans(n_clusters=k, random_state=42, n_init=10)
    l1 = km1.fit_predict(X_68_scaled)
    sil1 = silhouette_score(X_68_scaled, l1)
    
    # Skenario 2
    km2 = KMeans(n_clusters=k, random_state=42, n_init=10)
    l2 = km2.fit_predict(X_pca)
    sil2 = silhouette_score(X_pca, l2)
    
    results.append({
        'K': k,
        'Inertia_68': km1.inertia_,
        'Silhouette_68': sil1,
        'Inertia_PCA': km2.inertia_,
        'Silhouette_PCA': sil2,
    })

df_results = pd.DataFrame(results)
df_results

## 5.3.2 Visualisasi Komparasi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Elbow
axes[0].plot(df_results['K'], df_results['Inertia_68'], 'bo-', label='68 Fitur', linewidth=2)
axes[0].plot(df_results['K'], df_results['Inertia_PCA'], 'rs--', label='37 PCA', linewidth=2)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette
axes[1].plot(df_results['K'], df_results['Silhouette_68'], 'bo-', label='68 Fitur', linewidth=2)
axes[1].plot(df_results['K'], df_results['Silhouette_PCA'], 'rs--', label='37 PCA', linewidth=2)
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Bar chart komparasi K terbaik
best_68 = df_results.loc[df_results['Silhouette_68'].idxmax()]
best_pca = df_results.loc[df_results['Silhouette_PCA'].idxmax()]
bars = axes[2].bar(['68 Fitur', '37 PCA'],
                   [best_68['Silhouette_68'], best_pca['Silhouette_PCA']],
                   color=['steelblue', 'coral'], alpha=0.8)
axes[2].set_ylabel('Best Silhouette Score')
axes[2].set_title('Komparasi Skor Terbaik')
for bar, val, k in zip(bars,
                       [best_68['Silhouette_68'], best_pca['Silhouette_PCA']],
                       [best_68['K'], best_pca['K']]):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'Sil={val:.3f}\nK={int(k)}', ha='center', fontsize=10)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5.3.3 Tabel Ringkasan Akhir

In [ ]:
best_k_68 = int(best_68['K'])
best_k_pca = int(best_pca['K'])

ringkasan = pd.DataFrame({
    'Metrik': ['Jumlah Fitur', 'K Terbaik', 'Silhouette Score', 'Inertia', 'Status'],
    'Skenario 1 (68 Fitur)': [
        68,
        best_k_68,
        f"{best_68['Silhouette_68']:.4f}",
        f"{best_68['Inertia_68']:.2f}",
        'Tanpa Reduksi'
    ],
    'Skenario 2 (37 PCA)': [
        37,
        best_k_pca,
        f"{best_pca['Silhouette_PCA']:.4f}",
        f"{best_pca['Inertia_PCA']:.2f}",
        'Dengan Reduksi'
    ]
})
ringkasan

## 5.3.4 Interpretasi dan Kesimpulan

### Temuan Utama

1. **Silhouette Score**: Skenario dengan skor lebih tinggi menunjukkan cluster yang lebih terpisah (separated) dan compact (intra-cluster distance lebih kecil).

2. **Elbow Method**: K terbaik ditunjukkan oleh titik *elbow* — di mana penurunan inertia mulai melambat secara signifikan.

3. **Pengaruh PCA**: PCA dapat mengurangi noise dari fitur-fitur yang kurang informatif, sehingga cluster lebih jelas. Namun, PCA juga dapat membuang informasi penting jika komponen yang dibuang masih mengandung sinyal relevan.

### Rekomendasi

- Gunakan **Skenario 2 (37 PCA)** jika Silhouette Score-nya lebih tinggi → PCA efektif mereduksi noise.
- Gunakan **Skenario 1 (68 Fitur)** jika Silhouette Score-nya lebih tinggi → fitur asli lebih representatif.
- Dalam konteks analisis polutan, reduksi dimensi membantu mengisolasi sinyal polusi dari variasi noise satelit.

### Interpretasi Spasial

Kecamatan yang berada dalam cluster yang sama memiliki karakteristik polutan serupa — misalnya tingkat paparan rata-rata, pola fluktuasi harian, dan stabilitas konsentrasi yang mirip. Informasi ini berguna untuk:
- Pemetaan zona kualitas udara
- Penentuan prioritas intervensi lingkungan
- Perencanaan kebijakan pengendalian emisi